# KNPS 2026 불법시설물 변화탐지 exp01 (무영상 마스킹)

같은 위치의 전(pre)/후(post) 위성 RGB 패치(256x256) 한 쌍에서 **증축(new_building)** 과
**벌목(tree_removal)** 영역을 각각 폴리곤 목록으로 출력하는 참고 구현입니다. 성능 기준선이 아니라
**입력에서 제출 파일까지의 흐름과 제출 규격을 확인하는 최소 동작 예시**이며, 사용 의무는 없습니다.

- 입력: `AIF_INPUT_DIR` 아래 `pairs.csv`(`id` 열)와 `images/<id>/pre.png`, `images/<id>/post.png`
  (각 256x256 RGB 8bit, 두 영상은 같은 격자에 정렬되어 있습니다)
- 출력: `AIF_PREDICTION_PATH`에 `id,new_building,tree_removal` CSV 한 개. 두 열은 각각 그 클래스의 변화
  폴리곤 목록을 적은 JSON 문자열입니다. 폴리곤은 `[x, y]` 점의 목록(구멍 없음)이고 목록 안에 폴리곤을 여러 개
  담을 수 있습니다. 예: `[[[10,10],[30,10],[30,30],[10,30]]]`. 좌표는 후 영상 패치의 픽셀 좌표(원점 왼쪽 위
  모서리, x 오른쪽, y 아래, 실수 허용)이고 빈 문자열은 그 클래스의 변화가 없음을 뜻합니다. CSV 셀 안의 JSON 은
  `csv` 모듈이 인용 처리를 해 줍니다
- 동봉 가중치: `assets/model/unet_r18_cd.pt` (55MB). 추론 단계는 네트워크가 차단되므로 필요한
  파일은 모두 zip 안에 넣어야 합니다
- 의존성: `requirements.txt`로 선언합니다(폴리곤 변환에 shapely 를 씁니다). 채점 서버가 설치 단계에서 먼저 설치하므로 노트북 안에
  설치 셀을 두지 않았습니다. `%pip install -r requirements.txt`처럼 상대경로로 파일을 가리키는
  설치 줄은 설치 단계의 작업 폴더가 제출물 폴더가 아니어서 실패하므로 넣지 마시기 바랍니다
- 제출 규칙: zip 최상위에 `predict.ipynb`(이 파일)와 `requirements.txt`, 가중치는 `assets/`에 동봉

**로컬 실행**: `pairs.csv`와 `images/<id>/{pre,post}.png`를 담은 폴더를 만들고 환경변수로 넘기면 됩니다.
지정하지 않으면 `./input`을 읽고 `./prediction.csv`에 씁니다.
```
AIF_INPUT_DIR=<입력 폴더> AIF_PREDICTION_PATH=./prediction.csv jupyter nbconvert --to notebook --execute predict.ipynb
```

**라이선스**: segmentation_models_pytorch MIT, ResNet18 ImageNet 인코더(torchvision 원본) BSD-3,
파인튜닝 가중치와 이 코드는 대회 참가 목적으로 자유롭게 사용, 수정, 재학습하실 수 있습니다.
파생 제출물에 카피레프트나 별도 약관이 전파되지 않습니다. 출처는 `NOTICE`, 원문은 `LICENSE`에
있습니다. 이 베이스라인은 참고 구현이며 사용 의무가 없습니다.

In [ ]:
# [setup]
import csv
import io
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from shapely.geometry import MultiPolygon, Polygon, box
from shapely.ops import unary_union

INPUT_DIR = Path(os.environ.get("AIF_INPUT_DIR", "./input"))
PREDICTION_PATH = Path(os.environ.get("AIF_PREDICTION_PATH", "./prediction.csv"))
CKPT = Path("assets/model/unet_r18_cd.pt")      # 커널의 작업 폴더가 제출물 루트이므로 상대경로로 지정합니다

H, W = 256, 256
BATCH = 16
CLASSES = ("new_building", "tree_removal")       # 모델 출력 채널 1, 2 (0 은 배경)
MIN_AREA = 30            # 이 면적(px^2) 미만의 조각은 지웁니다 (학습 때와 같은 후처리)
MIN_POS_AREA = 20.0      # 조각을 지운 뒤 남은 총면적이 이 값 미만이면 빈 예측으로 냅니다 (채점 규칙과 동일)
SIMPLIFY_PX = 0.5        # 계단형 외곽선을 이 허용 오차로 단순화합니다 (위상 보존). 채점 허용 오차 1px 안입니다
NDIGITS = 2              # 좌표 소수 자릿수
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


def mask_to_polygons(mask: np.ndarray) -> str:
    """(H,W) 이진 마스크 -> 폴리곤 목록 JSON 문자열(폴리곤 = 점 목록, 구멍 없음). 변화가 없으면 빈 문자열.

    행 단위 런(run)을 화소 경계 사각형으로 만들어 합집합하면 마스크의 외곽선을 그대로 따르는 폴리곤이 나옵니다.
    좌표는 화소 모서리 기준(화소 (r, c) 는 [c, c+1] x [r, r+1])이라 채점 규약과 같습니다.
    """
    m = np.asarray(mask, dtype=bool)
    boxes = []
    for r in np.flatnonzero(m.any(axis=1)):
        pad = np.concatenate(([0], m[r].astype(np.int8), [0]))
        edges = np.flatnonzero(np.diff(pad))
        for s, e in zip(edges[::2], edges[1::2]):
            boxes.append(box(float(s), float(r), float(e), float(r + 1)))
    if not boxes:
        return ""
    g = unary_union(boxes)
    parts = list(g.geoms) if isinstance(g, MultiPolygon) else [g]
    parts = [p for p in parts if isinstance(p, Polygon) and p.area >= MIN_AREA]     # 작은 조각 제거
    if not parts or sum(p.area for p in parts) < MIN_POS_AREA:
        return ""
    out = []
    for p in parts:
        p = p.simplify(SIMPLIFY_PX, preserve_topology=True)
        if p.is_empty or p.area <= 0:
            continue
        # 바깥 경계만 씁니다(제출 형식에 구멍이 없습니다). 마지막 점은 첫 점과 같으므로 뺍니다
        out.append([[round(float(x), NDIGITS), round(float(y), NDIGITS)] for x, y in list(p.exterior.coords)[:-1]])
    return json.dumps(out, separators=(",", ":")) if out else ""

In [ ]:
# [inputs]
# 목록 파일을 재귀 탐색으로 찾고 그 부모 폴더를 입력 루트로 삼습니다 (껍데기 폴더 한 겹에 대비).
# 추론 대상은 목록 파일이 정본이며 images/ 폴더를 훑어 대상을 정하지 않습니다.
print("AIF_INPUT_DIR:", INPUT_DIR, sorted(p.name for p in INPUT_DIR.iterdir()) if INPUT_DIR.is_dir() else "(없음)")
lists = sorted(INPUT_DIR.rglob("pairs.csv"))
if not lists:
    # 입력이 풀린 폴더가 아니라 압축본 그대로 주어지는 경우에 대비합니다
    zs = sorted(INPUT_DIR.rglob("*.zip")) if INPUT_DIR.is_dir() else []
    if len(zs) == 1:
        import tempfile, zipfile
        dst = Path(tempfile.mkdtemp()) / "input"
        with zipfile.ZipFile(zs[0]) as z:
            z.extractall(dst)
        print(f"입력이 압축본이어서 풀어서 읽습니다: {zs[0].name} -> {dst}")
        lists = sorted(dst.rglob("pairs.csv"))
if not lists:
    raise SystemExit(f"pairs.csv 를 찾지 못했습니다: {INPUT_DIR}")
ROOT = lists[0].parent
with lists[0].open(encoding="utf-8-sig", newline="") as f:
    IDS = [row["id"].strip() for row in csv.DictReader(f) if row.get("id", "").strip()]
print(f"입력 루트 {ROOT}, 쌍 {len(IDS)}건")

In [ ]:
# [env]
print("python", sys.version.split()[0], "torch", torch.__version__)
print("cuda", torch.cuda.is_available(), "mps", getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available())

In [ ]:
# [model]
import segmentation_models_pytorch as smp


def load_model(device: str):
    model = smp.Unet(encoder_name="resnet18", encoder_weights=None, in_channels=6, classes=3)
    ck = torch.load(CKPT, map_location="cpu", weights_only=True)
    model.load_state_dict(ck["state_dict"])
    return model.to(device).eval()


def read_image(path: Path) -> np.ndarray:
    """파일을 직접 읽어 메모리 객체로 넘깁니다 (라이브러리에 경로를 맡기지 않습니다)."""
    with Image.open(io.BytesIO(path.read_bytes())) as im:
        arr = np.asarray(im.convert("RGB"))
    if arr.shape[:2] != (H, W):
        raise ValueError(f"{path}: 크기 {arr.shape[:2]} 가 {(H, W)} 와 다릅니다")
    return arr


@torch.no_grad()
def predict_batch(model, pres: list[np.ndarray], posts: list[np.ndarray], device: str) -> np.ndarray:
    """전/후 RGB 를 6채널로 이어 붙여 넣고 화소별 argmax 클래스(0/1/2)를 돌려줍니다."""
    x = np.stack([np.concatenate([(a / 255.0 - MEAN) / STD, (b / 255.0 - MEAN) / STD], axis=2)
                  for a, b in zip(pres, posts)]).astype(np.float32)
    x = torch.from_numpy(x).permute(0, 3, 1, 2).to(device)
    return torch.softmax(model(x), dim=1).argmax(1).cpu().numpy().astype(np.uint8)


# 가속기에서 1건을 시험 추론하고, 실패하면 CPU 로 전환합니다
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
try:
    MODEL = load_model(DEVICE)
    predict_batch(MODEL, [np.zeros((H, W, 3), np.uint8)], [np.zeros((H, W, 3), np.uint8)], DEVICE)
except Exception as e:  # noqa: BLE001
    print(f"{DEVICE} 시험 추론에 실패해({type(e).__name__}) CPU 로 전환합니다")
    DEVICE = "cpu"
    MODEL = load_model(DEVICE)
print("device", DEVICE, "params", sum(p.numel() for p in MODEL.parameters()))

In [ ]:
# [infer]
ROWS = []
t0 = time.time()
for s in range(0, len(IDS), BATCH):
    ids = IDS[s:s + BATCH]
    pres = [read_image(ROOT / "images" / i / "pre.png") for i in ids]
    posts = [read_image(ROOT / "images" / i / "post.png") for i in ids]
    labels = predict_batch(MODEL, pres, posts, DEVICE)
    for i, lab, pre, post in zip(ids, labels, pres, posts):
        # 전·후 영상 중 하나라도 완전히 검정인 화소는 채점 대상이 아닙니다.
        valid = np.any(pre != 0, axis=2) & np.any(post != 0, axis=2)
        lab = lab.copy()
        lab[~valid] = 0
        cells = [mask_to_polygons(lab == k) for k, _ in enumerate(CLASSES, start=1)]
        ROWS.append((i, *cells))
    if (s // BATCH) % 10 == 0 or s + BATCH >= len(IDS):
        print(f"  {min(s + BATCH, len(IDS))}/{len(IDS)}  {time.time() - t0:.1f}s", flush=True)
n_b = sum(1 for _, b, _ in ROWS if b)
n_t = sum(1 for _, _, t in ROWS if t)
print(f"추론을 완료했습니다: {len(ROWS)}건, 증축 양성 {n_b}건, 벌목 양성 {n_t}건, {time.time() - t0:.1f}s")

In [ ]:
# [save]
PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)
with PREDICTION_PATH.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", *CLASSES])
    w.writerows(ROWS)
print("저장했습니다:", PREDICTION_PATH, PREDICTION_PATH.stat().st_size, "bytes")

## 제출

아래 셀의 `<참여키>` 자리에 본인의 참여키를 넣고 실행하면 이 노트북이 들어 있는 폴더 전체가
제출됩니다. 참여키를 노트북에 적은 채로 공유하지 마시기 바랍니다. 참여키는 마이페이지에서
아래 경로를 따라 언제든 다시 확인하실 수 있습니다.

프로필 → 마이페이지 → 활동히스토리 탭 → 참가하신 주제의 키 복사

In [ ]:
%pip install -q --upgrade aifactory
%load_ext aifactory
%aifactory submit --api-key <참여키> --model-name exp01-nodata-mask